# Supplier Risk Intelligence Control Tower

Open-source model on AMD GPU, CrewAI orchestration, live event ingestion, graph memory, and Gradio dashboard.

In [ ]:

import os
import json
import asyncio
from pathlib import Path

# Local model endpoint served inside the AMD notebook environment.
os.environ.setdefault("BASE_URL", "http://localhost:8000/v1")
os.environ.setdefault("LOCAL_LLM_API_KEY", "abc-123")

MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct")
BASE_URL = os.environ["BASE_URL"]
API_KEY = os.environ["LOCAL_LLM_API_KEY"]

# Keep shared storage under the notebook workspace.
SHARED_DIR = Path("shared")
SHARED_DIR.mkdir(exist_ok=True)
print("Model:", MODEL_NAME)
print("Base URL:", BASE_URL)


In [ ]:

from src.config import SupplierProfile
from src.memory_store import EventMemory
from src.live_sources import build_live_query_bundle
from src.orchestration import assess_supplier, continuous_monitor
from src.app import build_ui

# The supplier twin keeps the demo grounded in one real procurement case.
supplier = SupplierProfile(
    name="TSMC",
    ticker="TSM",
    hq_country="Taiwan",
    operating_regions=["Taiwan", "Japan", "United States"],
    categories=["semiconductor manufacturing", "foundry", "advanced nodes"],
    critical_inputs=["EUV", "specialty chemicals", "advanced packaging"],
    notes="High exposure to geopolitics, logistics, and advanced-node supply continuity.",
)

memory = EventMemory(base_dir="shared", supplier_name=supplier.name.lower().replace(" ", "_"))
print(supplier)


In [ ]:

# Show the live query plan first so the operator can see what will be monitored.
query_map = build_live_query_bundle(supplier)
query_map


In [ ]:

# Run one full live cycle against the local open-source model.
result = await assess_supplier(
    supplier,
    memory,
    model_name=MODEL_NAME,
    base_url=BASE_URL,
    api_key=API_KEY,
)
print(json.dumps(result["snapshot"], indent=2, ensure_ascii=False))


In [ ]:

import pandas as pd

# Flatten the evidence trail into a readable table.
snapshot = result["snapshot"]
evidence = pd.DataFrame(snapshot["evidence"])
evidence.head(15)


In [ ]:

# Show the agent outputs without hiding them behind a single summary.
print("LANE BRIEFS\n")
print(json.dumps(result["lane_briefs"], indent=2, ensure_ascii=False))
print("\nSUMMARY\n")
print(json.dumps(result["summary"], indent=2, ensure_ascii=False))
print("\nCRITIQUE\n")
print(json.dumps(result["critique"], indent=2, ensure_ascii=False))


In [ ]:

# Run a short continuous monitor if you want repeated updates in the notebook.
monitor_results = await continuous_monitor(
    supplier,
    memory,
    cycles=2,
    pause_seconds=3,
    model_name=MODEL_NAME,
    base_url=BASE_URL,
    api_key=API_KEY,
)
[item["snapshot"]["overall_score"] for item in monitor_results]


In [ ]:

# Launch the dashboard so the runtime trail stays visible.
demo = build_ui()
demo.queue(default_concurrency_limit=4)
demo
